# 01 - Data Exploration
Goal: turn raw nflverse schedules into a clean, validated games table (2012-2025). 

## Setup
## 1. Load and filter
## 2. Grain: is each row one game? 
## 3. Types and ordering: build a kickoff time
## 4. Coverage: games per season and type
## 5. Validity: do columns mean what I think?
## 6. Standardize team abbreviations
## 7. Completeness of the columns I keep
## 8. The clean games table + data dictionary
## 9. Distributions: home margin, HFA, σ, the market
## Decisions made
## Open questions

## Setup

In [1]:
import nflreadpy as nfl
import pandas as pd

## 1. Load and filter

In [2]:
raw = nfl.load_schedules().to_pandas()
raw.shape

(7548, 46)

In [3]:
def filter_seasons(df, first=2012, last = 2025):
    """Return rows whose season is between first and last, inclusive"""
    return df[df["season"].between(first, last)]

games = filter_seasons(raw).copy()
games.shape

(3829, 46)

## 2. Grain

In [4]:
n_dupes = games["game_id"].duplicated().sum()
print(f"Duplicate game_ids: {n_dupes}")
assert n_dupes == 0, "Duplicate game_ids found"

Duplicate game_ids: 0


## 3. Types and ordering

In [ ]:
"""
The gameday and gametime columns are joined together as a real datetime under the new 
column "Kickoff." The values in the dataframe are sorted and renumbered. 
"""
games["Kickoff"] = pd.to_datetime(games["gameday"] + " " + games["gametime"])
games = games.sort_values("Kickoff").reset_index(drop=True)
games[["game_id", "week", "Kickoff"]].head(10)


,game_id,week,Kickoff
0,2012_01_DAL_NYG,1,2012-09-05 20:30:00
1,2012_01_IND_CHI,1,2012-09-09 13:00:00
2,2012_01_PHI_CLE,1,2012-09-09 13:00:00
3,2012_01_STL_DET,1,2012-09-09 13:00:00
4,2012_01_MIA_HOU,1,2012-09-09 13:00:00
5,2012_01_ATL_KC,1,2012-09-09 13:00:00
6,2012_01_JAX_MIN,1,2012-09-09 13:00:00
7,2012_01_WAS_NO,1,2012-09-09 13:00:00
8,2012_01_BUF_NYJ,1,2012-09-09 13:00:00
9,2012_01_NE_TEN,1,2012-09-09 13:00:00


In [6]:
weeks_in_order = games.groupby("season")["week"].is_monotonic_increasing.all()
print(f"Weeks never go backward within a season: {weeks_in_order}")
assert weeks_in_order

Weeks never go backward within a season: True


## 4. Coverage

In [7]:
counts = pd.crosstab(games["season"], games["game_type"])
counts

game_type,CON,DIV,REG,SB,WC
season,,,,,
2012,2,4,256,1,4
2013,2,4,256,1,4
2014,2,4,256,1,4
2015,2,4,256,1,4
2016,2,4,256,1,4
2017,2,4,256,1,4
2018,2,4,256,1,4
2019,2,4,256,1,4
2020,2,4,256,1,6


- This table reveals 3 items to note: 
    1. The playoff team expansion in 2020, increasing the number of WC games to 6
    2. The 17-game-schedule expansion in 2021, increasing the REG game count to 272
    3. Only 271 games in 2022 due to the no-contest ruling of the Bengals-Bills game in which Damar Hamlin suffered a severe injury

In [14]:
# Win percent of the home team in the regular season vs. playoff
games["Is_Playoff"] = games["game_type"] != "REG"
games["Home_Win"] = (games["result"] > 0).astype(int)
print(games.groupby("Is_Playoff")["Home_Win"].count())
print(games.groupby("Is_Playoff")["Home_Win"].mean())

Is_Playoff
False    3663
True      166
Name: Home_Win, dtype: int64
Is_Playoff
False    0.551188
True     0.626506
Name: Home_Win, dtype: float64


**Finding:** In true home games (nuetral sites excluded), home teams won 55.1% of 3663 regular season games and 62.7% of 166 playoff games. The playoff sample is small, so the gap is suggestive rather than conclusive. Playoff home field goes to the higher seed, so part of the gap likely reflects team strenth rather than home advantage. 

**Decision (provisional):**
- Include playoff games in Elo rating updates. 
- Apply home-feild advantage to playoff games at the higher seed's stadiu; none at the nuetral sites (e.g., Super Bowls).
- Estiamte the size of home-feild advantage from regular season, non-nuetral games only, where the balanced schedule measn home and away teams are equally strong on average. 
- Revisit in Phase 1: tune with and without playoff games and compare Brier scores. 

## 5. Validity

In [ ]:
# Check the results column
calc_result = games["home_score"] - games["away_score"]
mismatches = (games["result"] != calc_result).sum()
print(f"result mismatches: {mismatches}")
assert mismatches == 0, "result != home_score - away_score"

result mismatches: 0
